# How do you score a match between two lists of numbers?


Your music app just played you a song by a band you've never heard of, and called it a match for your taste. Somewhere, two descriptions got compared and collapsed into one verdict.

A song is not one number. It's loud, fast, sad, bright, a dozen qualities at once. Your taste is that same kind of tangle. No single ruler works here: the ruler that measures loud says nothing about sad.

Say taste = [2, 1] (loud-ness, sad-ness) and a song = [1, 3]. Before you run any code, try to answer this on paper: how would you combine those four numbers into one match score? It's harder than it looks.


In [ ]:
import numpy as np

taste = np.array([2.0, 1.0])   # [loud, sad]
song = np.array([1.0, 3.0])    # [loud, sad]

print("taste:", taste)
print("song: ", song)


Adding everything up lets one loud feature drown the rest. Waiting for an exact match never happens, because no two real things agree on every quality. Here's the move that works: treat each list as an arrow, then multiply the two arrows entry by entry and add the products.

For taste = [2, 1] and song = [1, 3], that's 2 x 1 + 1 x 3 = 5. Before you run the next cell, predict that number yourself, then check it.


In [ ]:
def hand_dot(a, b):
    total = 0.0
    for x, y in zip(a, b):
        total += x * y
    return total

manual = hand_dot(taste, song)
builtin = np.dot(taste, song)
print("manual multiply-and-add:", manual)
print("np.dot:                 ", builtin)

assert manual == builtin == 5.0


Now meet the instrument: one probe arrow (your taste), and four candidate arrows holding still, each the same length. As the probe sweeps all the way around a full circle, does the score against any candidate ever go negative? Answer yes or no before running the next cell.


In [ ]:
angles = np.array([0, 45, 90, 135, 180, 225, 270, 315])
probe_length = 1.0

candidates = {
    "candidate_A": np.array([1.0, 0.0]),
    "candidate_B": np.array([0.0, 1.0]),
    "candidate_C": np.array([-1.0, 0.0]),
    "candidate_D": np.array([0.7, 0.7]),
}

for deg in angles:
    rad = np.deg2rad(deg)
    probe = probe_length * np.array([np.cos(rad), np.sin(rad)])
    scores = {name: round(float(np.dot(probe, c)), 2) for name, c in candidates.items()}
    print(f"probe at {deg:3d} deg:", scores)


In [ ]:
all_scores = []
for deg in angles:
    rad = np.deg2rad(deg)
    probe = probe_length * np.array([np.cos(rad), np.sin(rad)])
    for c in candidates.values():
        all_scores.append(np.dot(probe, c))

assert min(all_scores) < 0, "somewhere in the sweep a score should go negative"
assert max(all_scores) > 0, "somewhere in the sweep a score should go positive"
print("lowest score: ", round(float(min(all_scores)), 2))
print("highest score:", round(float(max(all_scores)), 2))


So yes, it does go negative. The candidates never moved. While every arrow held the same length, direction was all the score ever responded to: point along a candidate and the score climbs, point across and it dies to zero, point against and it goes negative.

So a big score means two things are the same, right? Watch what length does next.


Here are two candidates at fixed directions: one well-aligned with the probe, one pointed further off. Right now, which one scores higher? Decide before running the next cell, then check.


In [ ]:
probe = np.array([1.0, 0.0])

well_aligned = np.array([0.9, 0.1])
off_direction = np.array([0.5, 0.9])

score_aligned = round(float(np.dot(probe, well_aligned)), 2)
score_off = round(float(np.dot(probe, off_direction)), 2)

print("well-aligned score:  ", score_aligned)
print("off-direction score:", score_off)

assert score_aligned > score_off


Now stretch off_direction to three times its length, direction unchanged. Before running the next cell: will it out-score well_aligned, or is direction too far behind for length to make up?


In [ ]:
stretched = off_direction * 3.0
score_stretched = round(float(np.dot(probe, stretched)), 2)

print("off-direction score after 3x stretch:", score_stretched)
print("well-aligned score (unchanged):      ", score_aligned)

assert score_stretched > score_aligned, "a longer arrow just out-scored a better-aligned one"


Look at what you just did by hand. Point along a candidate and the sum climbs. Point across and it dies to zero. Point against and it goes negative. One multiply-and-add, and it behaves like a meter with agree, ignore, and oppose on the dial.

**The dot product is a similarity meter.**

In symbols: a . b = a1*b1 + a2*b2 + ... For taste = [2, 1] and song = [1, 3], a . b = 2 x 1 + 1 x 3 = 5, the same recipe wearing its symbol.


In [ ]:
probe = np.array([2.0, 1.0])
aligned = probe.copy()
orthogonal = np.array([-1.0, 2.0])
opposite = -probe

score_aligned = round(float(np.dot(probe, aligned)), 2)
score_orthogonal = round(float(np.dot(probe, orthogonal)), 2)
score_opposite = round(float(np.dot(probe, opposite)), 2)

print("aligned score:   ", score_aligned)
print("orthogonal score:", score_orthogonal)
print("opposite score:  ", score_opposite)

assert score_aligned > 0
assert score_orthogonal == 0
assert score_opposite < 0


Rebuild the meter from memory: two arrows go in, so what two operations turn them into the score? Multiply entrywise, then add.

Now scale it up. Your taste is the probe. The catalog is a shelf of candidates, thousands of arrows long. The machine in the data center runs multiply-and-add across the whole shelf and hands you the top score.


In [ ]:
rng = np.random.default_rng(0)

taste = np.array([2.0, 1.0])
catalog_size = 500
catalog = rng.normal(0, 1, size=(catalog_size, 2))

planted_index = 250
catalog[planted_index] = taste * 5.0   # a song that shares taste's direction, and then some

scores = catalog @ taste
winner = int(np.argmax(scores))

print("planted index:", planted_index)
print("winner index: ", winner)
print("winner score: ", round(float(scores[winner]), 2))

assert winner == planted_index


You just saw a longer arrow win, even without a direction change. Before you run the next cell: point a candidate called Night Drive exactly along the probe, at length 1.0, so its score reads a clean number. Then double Night Drive's length exactly, same direction. Does the score double exactly, or just drift up?


In [ ]:
probe = np.array([1.0, 0.0])
night_drive = np.array([1.0, 0.0])   # same direction as the probe, length 1.0

score_1x = np.dot(probe, night_drive)
score_2x = np.dot(probe, night_drive * 2.0)

print("score at length 1.0:", round(float(score_1x), 2))
print("score at length 2.0:", round(float(score_2x), 2))

assert score_2x == 2.0 * score_1x


It doubles, exactly: 1.0 becomes 2.0, to the last digit. That's not a fact about this one candidate. Double any input arrow and you double the score, always, because the dot product is linear in each arrow you feed it. That clean scaling is the crack: a candidate can win purely by being loud, whether or not its direction agrees.


In [ ]:
rng2 = np.random.default_rng(1)

for _ in range(5):
    a = rng2.normal(size=3)
    b = rng2.normal(size=3)
    scale = rng2.uniform(0.5, 3.0)
    lhs = np.dot(a * scale, b)
    rhs = scale * np.dot(a, b)
    assert np.isclose(lhs, rhs)

print("checked 5 random (vector, scale) pairs: dot(a * scale, b) == scale * dot(a, b), every time")


Take it with you. A song and a taste folded into arrows, and multiply-and-add collapsed them into one score: a meter with a crack in it.

Same meter, different aisle: the arithmetic doesn't care whether the features are named loud and sad, or python-skill and night-shifts. Score yourself against a job posting the same way, matching entries, multiplying, adding.


In [ ]:
you = np.array([4.0, 5.0, 2.0])       # [python skill, communication, night-shift willingness]
posting = np.array([5.0, 3.0, 0.0])

fit_score = np.dot(you, posting)
print("fit score:", round(float(fit_score), 2))

assert np.dot(you, posting) == np.dot(posting, you)


## For the walk home

What is the cheapest fix that keeps the direction and forgets the size?

One thing to try next: pick two candidates that point in the same direction but have very different lengths, score them against a probe, and experiment until you find a change that makes their scores equal no matter how long either one is.

Next time your app hands you a stranger's song that's exactly right, you'll know the number that said ship it, and you'll know that number can be shouted over.
